# 03 — STL Deep Dive

## Why This Matters for DSA
The Standard Template Library is not a bonus feature of C++ — it IS how you write C++ for data structures and algorithms. Every tree, graph, and DP problem in this course leans on `vector`, `map`, `set`, `priority_queue`, and the `<algorithm>` header. If you don't know these cold — their real complexity, their failure modes, their iterator rules — you will write code that compiles, looks reasonable, and is either wrong or an order of magnitude slower than it should be. This notebook is the reference you come back to every time you forget whether `map::operator[]` inserts, or whether `list` iterators support `+`.

## Prerequisites
- Completion of `01_Pointers_and_References` and `02_Classes_and_Memory` (or equivalent knowledge of pointers, references, and object lifetime)
- Comfort with basic C++ syntax: loops, functions, templates at a "I can read them" level

## Learning Objectives
By the end of this notebook, you will be able to:
- Explain the real time complexity of every common operation on `vector`, `deque`, `list`, `set`, `map`, and their unordered/multi variants
- Choose the correct container for a given access pattern instead of defaulting to `vector` or `map` out of habit
- Avoid the specific bugs that come from iterator invalidation and `map::operator[]`'s silent insertion
- Use the `<algorithm>` header fluently: `sort`, `binary_search`, `lower_bound`/`upper_bound`, `next_permutation`, the erase-remove idiom
- Write and read lambdas confidently, including capture semantics and `mutable`
- Supply custom comparators to `sort`, `set`, and `priority_queue` when the default ordering isn't what you need

## Checklist Before Moving On
- [ ] I can state the amortized complexity of `vector::push_back` and explain WHY it's amortized, not exact
- [ ] I know which containers invalidate iterators on insert/erase, and which don't
- [ ] I know the difference between `map::find` and `map::operator[]` and why mixing them up causes bugs
- [ ] I can write a custom comparator for `sort` and for `priority_queue` from memory
- [ ] I understand the difference between capturing `[x]` and `[&x]` in a lambda
- [ ] I know when to reach for `unordered_map` vs `map`

## Table of Contents
1. `std::vector` — Growth, Capacity, and the Amortized Cost Argument
2. `std::vector` — Insert, Erase, and the Erase-Remove Idiom
3. `std::pair` and `std::tuple`
4. `std::string` Deep Dive
5. `std::array` and `std::deque`
6. `std::list` and `std::forward_list`
7. Container Adapters — `stack`, `queue`, `priority_queue`
8. `std::set` / `std::map` and Their Unordered Counterparts
9. Ordered Range Queries — `lower_bound` / `upper_bound` on `set` and `map`
10. Iterators — Categories, Validity, and `advance`
11. The `<algorithm>` Header
12. Lambdas and `std::function`
13. `std::bitset` and Custom Comparators for User-Defined Types
14. Phase Checkpoint, Complexity Cheat Sheet, and Answer Key


## 1. `std::vector` — Growth, Capacity, and the Amortized Cost Argument

### The Why
`vector` is the default container for a reason: contiguous memory means cache-friendly iteration, and `operator[]` is O(1). But `push_back` has a cost story that trips people up in interviews — "is `push_back` O(1)?" has a real answer that isn't just yes.

### Core Mechanism
- `size()` is how many elements are actually stored. `capacity()` is how much memory is allocated — capacity is always ≥ size.
- When `push_back` would exceed capacity, the vector allocates a NEW, bigger block (typically double the old capacity), copies/moves every existing element into it, then frees the old block.
- That one `push_back` is O(n). But it only happens roughly every log₂(n) pushes — averaged ("amortized") over all pushes, each one costs O(1).
- `reserve(n)` lets you pre-allocate capacity when you know the size upfront, avoiding repeated reallocation entirely.
- A `vector<vector<int>>` is the standard way to build a 2D grid — each row is its own independently allocated `vector<int>`.

### Common Pitfalls
- Assuming `push_back` is always O(1) — it's O(1) amortized, not O(1) worst case. If a single call must be fast (e.g. a hard real-time constraint), this matters.
- Forgetting `reserve()` exists, then being surprised a tight loop with millions of pushes is slower than expected due to repeated reallocation.


In [ ]:
%%writefile temp.cpp
#include <vector>
#include <iostream>
using namespace std;

int main() {
    vector<int> v;                      // empty vector, size=0, capacity=0
    cout << "size=" << v.size() << " capacity=" << v.capacity() << "\n";

    for (int i = 0; i < 5; i++) {
        v.push_back(i * 10);
        cout << "after push_back(" << i * 10 << "): size=" << v.size()
             << " capacity=" << v.capacity() << "\n";
    }

    // capacity grows by doubling (implementation-defined, but this is the common strategy)
    // this amortizes push_back to O(1) average, even though a single push_back
    // that triggers reallocation is O(n)

    vector<int> preallocated;
    preallocated.reserve(100);          // allocate capacity upfront, size stays 0
    cout << "reserved: size=" << preallocated.size()
         << " capacity=" << preallocated.capacity() << "\n";

    // 2D vector: vector of vectors, each row independently allocated
    vector<vector<int>> grid(3, vector<int>(4, 0)); // 3 rows, 4 cols, all zero
    grid[1][2] = 99;
    for (auto &row : grid) {
        for (int x : row) cout << x << " ";
        cout << "\n";
    }

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 2. `std::vector` — Insert, Erase, and the Erase-Remove Idiom

### The Why
`push_back`/`pop_back` are cheap because they only touch the end. Insert or erase ANYWHERE ELSE is expensive, because every element after that point has to shift. Knowing this changes which container you reach for.

### Core Mechanism
- `insert(it, val)` and `erase(it)` at an arbitrary position are O(n) — everything from that position to the end shifts by one slot.
- `pop_back()` is O(1) — no shifting, just a size decrement (and a destructor call).
- **Erase-remove idiom:** `vector` has no "remove all elements equal to X" method that actually shrinks the container. `std::remove` moves the elements you want to KEEP to the front and returns an iterator to the new logical end — but the vector's `size()` doesn't change yet. You must follow it with `erase()` to actually cut off the leftover tail. This two-step is the standard, correct way to delete-by-value from a vector.

### Common Pitfalls
- Calling `remove()` alone and assuming the vector shrank — it doesn't. `size()` is unchanged until you call `erase()` too.
- Holding onto an iterator across a `push_back`/`insert` that might reallocate. Any reallocation invalidates EVERY iterator, pointer, and reference into the vector — not just ones near the change.


In [ ]:
%%writefile temp.cpp
#include <vector>
#include <iostream>
#include <algorithm>
using namespace std;

int main() {
    vector<int> v{10, 20, 30, 40, 50};

    // insert at arbitrary position: O(n) because everything after must shift
    v.insert(v.begin() + 2, 99);        // {10,20,99,30,40,50}
    for (int x : v) cout << x << " ";
    cout << "\n";

    // erase at arbitrary position: also O(n), same shifting cost
    v.erase(v.begin() + 1);             // remove the 20
    for (int x : v) cout << x << " ";
    cout << "\n";

    // erase-remove idiom: the ONLY correct way to remove all elements matching a value
    // remove() doesn't actually shrink the vector -- it moves "kept" elements to the front
    // and returns an iterator to the new logical end. erase() then trims the rest.
    vector<int> nums{1, 2, 3, 2, 4, 2, 5};
    nums.erase(remove(nums.begin(), nums.end(), 2), nums.end());
    for (int x : nums) cout << x << " ";
    cout << "\n";

    // iterator invalidation trap: push_back can reallocate, invalidating ALL iterators
    vector<int> danger{1, 2, 3};
    auto it = danger.begin();
    (void)it; // silence "unused" warning -- we intentionally never read it after invalidation
    danger.push_back(4);   // may reallocate -> `it` could now be dangling
    // *it is undefined behavior here if reallocation happened -- never do this

    // pop_back is O(1) -- only shrinks size, no shifting needed
    v.pop_back();
    cout << "after pop_back, size=" << v.size() << "\n";

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 3. `std::pair` and `std::tuple`

### The Why
You'll reach for `pair` constantly — coordinates, (node, distance) in Dijkstra, (value, index) when you need both. `tuple` generalizes this to more than two values.

### Core Mechanism
- `pair<T1, T2>` holds exactly two values, accessed via `.first` and `.second`.
- Pairs compare **lexicographically**: `.first` is compared first; `.second` only breaks ties. This is why `vector<pair<int,int>>` sorts "by first, then by second" with zero extra code.
- `tuple<T1, T2, T3, ...>` holds any number of values of any types, accessed with `get<index>(t)`.
- **Structured bindings** (`auto [a, b] = pair_or_tuple;`, C++17) unpack a pair or tuple into named variables — this is what makes `for (auto &[key, val] : myMap)` readable instead of `.first`/`.second` everywhere.

### Common Pitfalls
- Forgetting pairs sort lexicographically and being surprised when a "sort by distance" vector of `pair<int,int>` also silently tie-breaks by the first element.


In [ ]:
%%writefile temp.cpp
#include <utility>
#include <tuple>
#include <iostream>
#include <vector>
#include <algorithm>
using namespace std;

int main() {
    // pair: exactly two values, accessed via .first / .second
    pair<int, string> p = {1, "apple"};
    cout << p.first << " " << p.second << "\n";

    // make_pair lets the compiler deduce types
    auto p2 = make_pair(2, "banana");
    cout << p2.first << " " << p2.second << "\n";

    // pairs compare lexicographically: first compared first, second breaks ties
    vector<pair<int,int>> pts{{2,1}, {1,5}, {1,2}};
    sort(pts.begin(), pts.end());   // sorts by .first, then by .second on ties
    for (auto &pr : pts) cout << "(" << pr.first << "," << pr.second << ") ";
    cout << "\n";

    // tuple: any number of values, any types
    tuple<int, string, double> t = {3, "cherry", 4.5};
    cout << get<0>(t) << " " << get<1>(t) << " " << get<2>(t) << "\n";

    // structured bindings (C++17): unpack pair/tuple into named variables
    auto [id, name, price] = t;
    cout << "id=" << id << " name=" << name << " price=" << price << "\n";

    for (auto &[a, b] : pts) {   // works for pairs too, very common in map iteration
        cout << a << "-" << b << " ";
    }
    cout << "\n";

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 4. `std::string` Deep Dive

### The Why
Nearly every string problem needs at least one of: substring extraction, searching, splitting, or number conversion. `stringstream` in particular is the standard tool for splitting on whitespace or a custom delimiter — most people don't discover it and instead hand-roll fragile parsing loops.

### Core Mechanism
- `substr(pos, len)` — `len` is optional; omit it to get everything from `pos` to the end.
- `find(sub)` returns `string::npos` — a huge sentinel constant, NOT `-1` — when the substring isn't found. Always compare against `string::npos`, never against `-1`.
- `to_string(n)` converts number → string. `stoi`/`stod`/`stol` convert string → number.
- `stringstream ss(s); ss >> word;` in a `while` loop splits on whitespace, one token at a time.
- `getline(ss, token, delimiter)` splits on any custom single-character delimiter (like a comma for CSV).
- `std::string` is mutable — `s[0] = 'H'` works directly, and algorithms like `reverse()` operate on it in place, just like a `vector<char>`.

### Common Pitfalls
- Comparing `find()`'s result to `-1` instead of `string::npos` — `npos` is an unsigned type, so this comparison can silently do the wrong thing.
- Trying to split a string manually with index math when `stringstream` already solves it cleanly.


In [ ]:
%%writefile temp.cpp
#include <string>
#include <iostream>
#include <sstream>
#include <algorithm>
using namespace std;

int main() {
    string s = "Hello, World!";

    // substr(pos, len) -- len is optional, defaults to "rest of string"
    cout << s.substr(7) << "\n";        // "World!"
    cout << s.substr(0, 5) << "\n";     // "Hello"

    // find returns string::npos (a huge sentinel value) if not found -- always check for it
    size_t pos = s.find("World");
    if (pos != string::npos) cout << "found at index " << pos << "\n";

    // string concatenation and comparison work like you'd expect
    string a = "abc", b = "abd";
    cout << (a < b ? "a < b" : "a >= b") << "\n";  // lexicographic comparison

    // converting number <-> string
    string numStr = to_string(42);
    int backToInt = stoi("123");
    double backToDouble = stod("3.14");
    cout << numStr << " " << backToInt << " " << backToDouble << "\n";

    // stringstream: the standard way to split a string by whitespace
    string sentence = "the quick brown fox";
    stringstream ss(sentence);
    string word;
    while (ss >> word) {
        cout << "[" << word << "] ";
    }
    cout << "\n";

    // splitting by a custom delimiter (comma) using getline on a stringstream
    string csv = "a,bb,ccc";
    stringstream ss2(csv);
    string token;
    while (getline(ss2, token, ',')) {
        cout << token << "(" << token.size() << ") ";
    }
    cout << "\n";

    // strings are mutable and support in-place modification, unlike some languages
    string mutable_s = "hello";
    mutable_s[0] = 'H';
    reverse(mutable_s.begin(), mutable_s.end());
    cout << mutable_s << "\n";

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 5. `std::array` and `std::deque`

### The Why
`std::array` is what you use instead of a raw C-style array when the size is fixed and known at compile time — you get bounds-aware methods without losing stack allocation. `std::deque` is the answer whenever you need fast insertion at BOTH ends, which `vector` cannot give you.

### Core Mechanism
- `array<T, N>` — size `N` is a compile-time constant, part of the type itself. `.size()` is O(1) and known before the program even runs.
- `deque` (double-ended queue) supports O(1) `push_front`/`pop_front` in addition to `push_back`/`pop_back` — `vector` only has the "back" operations at O(1); its `push_front` would be O(n).
- `deque` still supports random access via `operator[]`, just with a different internal layout (typically a sequence of fixed-size chunks, not one contiguous block).

### Common Pitfalls
- Reaching for `vector` and manually simulating a front-queue with index tracking, when `deque` already does this correctly and efficiently.


In [ ]:
%%writefile temp.cpp
#include <array>
#include <deque>
#include <iostream>
using namespace std;

int main() {
    // std::array: fixed-size, stack-allocated (when local), size known at compile time
    array<int, 5> arr = {1, 2, 3, 4, 5};
    cout << "array size=" << arr.size() << " front=" << arr.front() << " back=" << arr.back() << "\n";
    // arr.size() is O(1) and known at compile time -- unlike a raw C array, array knows its own size

    // deque (double-ended queue): O(1) push/pop at BOTH ends, unlike vector (only fast at back)
    deque<int> dq;
    dq.push_back(1);
    dq.push_back(2);
    dq.push_front(0);
    dq.push_front(-1);
    for (int x : dq) cout << x << " ";   // -1 0 1 2
    cout << "\n";

    dq.pop_front();
    dq.pop_back();
    for (int x : dq) cout << x << " ";   // 0 1
    cout << "\n";

    // deque also supports random access via [] like vector, just with different internal layout
    cout << "dq[0]=" << dq[0] << "\n";

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 6. `std::list` and `std::forward_list`

### The Why
These are the STL's actual linked lists. They're rarely the right default choice (contiguous containers usually win on cache performance even when the Big-O looks worse), but they matter for two things: understanding the trade-off, and `splice()`, which lets you move nodes between lists in O(1) with zero copying.

### Core Mechanism
- `list` is doubly linked: O(1) insert/erase at any position **you already have an iterator to** — but reaching that position by index is O(n), since there's no random access.
- `list` has no `operator[]` at all — this is intentional, to stop you from writing accidentally-O(n²) code that indexes into a list in a loop.
- `splice()` moves a range of nodes from one list into another by just relinking pointers — O(1), no element copying, no invalidation of iterators to the moved elements.
- `forward_list` is singly linked — even less memory overhead than `list`, but you can only iterate forward (`++`, never `--`), and there's no `push_back()` or `size()` by design.

### Common Pitfalls
- Using `list` as a default container and then indexing into it in a loop (`for (int i = 0; i < lst.size(); i++) lst[i]` — this doesn't even compile, but the pattern of "walk to position i" done repeatedly is the O(n²) trap people fall into with `advance()`).


In [ ]:
%%writefile temp.cpp
#include <list>
#include <forward_list>
#include <iostream>
using namespace std;

int main() {
    // std::list: doubly linked list. O(1) insert/erase ANYWHERE if you have an iterator,
    // but O(n) to reach that position (no random access, no operator[])
    list<int> lst{1, 2, 3, 4, 5};

    auto it = lst.begin();
    advance(it, 2);              // move iterator 2 steps forward -- O(n) for list
    lst.insert(it, 99);          // O(1) once positioned: {1,2,99,3,4,5}

    for (int x : lst) cout << x << " ";
    cout << "\n";

    lst.erase(it);               // erase the element `it` points to (the original 3)
    for (int x : lst) cout << x << " ";
    cout << "\n";

    // splice: move elements between lists in O(1), no copying -- this is list's superpower
    list<int> a{1, 2, 3};
    list<int> b{100, 200};
    a.splice(a.begin(), b);      // move all of b to the front of a, b becomes empty
    for (int x : a) cout << x << " ";
    cout << "| b size=" << b.size() << "\n";

    // forward_list: singly linked, even less overhead than list, but ONLY forward iteration
    forward_list<int> fl{1, 2, 3};
    fl.push_front(0);            // only push_front exists -- no push_back, no size()
    for (int x : fl) cout << x << " ";
    cout << "\n";

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 7. Container Adapters — `stack`, `queue`, `priority_queue`

### The Why
These three are used in nearly every DFS, BFS, and greedy/heap problem you'll write. They're called "adapters" because they aren't independent containers — they wrap another container (`deque` by default) and restrict its interface to a specific access pattern.

### Core Mechanism
- `stack`: LIFO. `.push()`, `.pop()`, `.top()`. Built on `deque` by default.
- `queue`: FIFO. `.push()`, `.pop()`, `.front()`, `.back()`. Also `deque`-backed by default.
- `priority_queue`: max-heap by default — `.top()` is always the largest element. `.push()` and `.pop()` are O(log n).
- **Min-heap:** pass `greater<T>` as the third template argument: `priority_queue<int, vector<int>, greater<int>>`.
- **Custom comparator:** for anything beyond a primitive type, or a non-default ordering, pass a lambda and use `decltype(cmp)` as the comparator type — the constructor then takes the comparator instance itself, since a lambda's type is compiler-generated and has no other name you could write down.

### Common Pitfalls
- Forgetting `priority_queue` is a MAX-heap by default and being surprised `.top()` isn't the smallest element.
- Writing the custom-comparator lambda backwards — remember the comparator answers "does A have LOWER priority than B", so for a min-heap-on-second-value, you return `a.second > b.second` (A is lower priority when its value is bigger).


In [ ]:
%%writefile temp.cpp
#include <stack>
#include <queue>
#include <vector>
#include <iostream>
using namespace std;

int main() {
    // stack: LIFO. Built on top of deque by default (an "adapter", not its own container)
    stack<int> st;
    st.push(1); st.push(2); st.push(3);
    cout << "stack top=" << st.top() << "\n";
    st.pop();
    cout << "stack top after pop=" << st.top() << "\n";

    // queue: FIFO. Also a deque adapter by default
    queue<int> q;
    q.push(1); q.push(2); q.push(3);
    cout << "queue front=" << q.front() << " back=" << q.back() << "\n";
    q.pop();
    cout << "queue front after pop=" << q.front() << "\n";

    // priority_queue: max-heap by default
    priority_queue<int> maxHeap;
    maxHeap.push(3); maxHeap.push(1); maxHeap.push(4); maxHeap.push(1); maxHeap.push(5);
    cout << "max-heap order: ";
    while (!maxHeap.empty()) {
        cout << maxHeap.top() << " ";
        maxHeap.pop();
    }
    cout << "\n";

    // min-heap: pass greater<int> as the comparator (3rd template argument)
    priority_queue<int, vector<int>, greater<int>> minHeap;
    minHeap.push(3); minHeap.push(1); minHeap.push(4); minHeap.push(1); minHeap.push(5);
    cout << "min-heap order: ";
    while (!minHeap.empty()) {
        cout << minHeap.top() << " ";
        minHeap.pop();
    }
    cout << "\n";

    // custom comparator with a lambda: needs decltype since lambda types are unique/unnamed
    auto cmp = [](pair<int,int> a, pair<int,int> b) {
        return a.second > b.second;  // smaller .second has higher priority (min-heap on .second)
    };
    priority_queue<pair<int,int>, vector<pair<int,int>>, decltype(cmp)> pq(cmp);
    pq.push({1, 50});
    pq.push({2, 10});
    pq.push({3, 30});
    cout << "custom comparator order: ";
    while (!pq.empty()) {
        cout << "(" << pq.top().first << "," << pq.top().second << ") ";
        pq.pop();
    }
    cout << "\n";

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 8. `std::set` / `std::map` and Their Unordered Counterparts

### The Why
This is where the single most common STL bug lives: `map::operator[]` silently INSERTS a default-constructed value if the key doesn't already exist. If you only ever use `[]` to check existence, you will slowly corrupt your own map without any error or warning.

### Core Mechanism
- `set<T>`: sorted, unique elements. Backed by a red-black tree (see the README's Advanced Trees section) — O(log n) insert/find/erase, and iterating always visits elements in sorted order.
- `multiset<T>`: like `set`, but allows duplicate elements. `.count(x)` tells you how many copies exist.
- `map<K,V>`: sorted key-value pairs, same red-black tree backing, O(log n) operations, iterates in sorted-by-key order.
- `unordered_set` / `unordered_map`: hash-table backed. O(1) average for insert/find/erase, O(n) worst case (pathological hash collisions), and **no ordering guarantee whatsoever** — never rely on iteration order.
- **The `operator[]` trap:** `map["key"]` — if `"key"` isn't present, this INSERTS it with a default-constructed value (0 for `int`, `""` for `string`) and returns a reference to that new entry. Use `.find(key) == m.end()` or `.count(key)` to check existence without mutating the map.

### When to Use Which
- Need sorted order or range queries (`lower_bound`/`upper_bound`)? → `set` / `map`.
- Need the fastest possible average lookup and don't care about order at all? → `unordered_set` / `unordered_map`.


In [ ]:
%%writefile temp.cpp
#include <set>
#include <map>
#include <unordered_set>
#include <unordered_map>
#include <iostream>
using namespace std;

int main() {
    // set: sorted, unique elements, backed by a red-black tree -- O(log n) insert/find/erase
    set<int> s{5, 3, 1, 4, 1, 5, 9};   // duplicates silently dropped
    for (int x : s) cout << x << " ";  // always iterates in sorted order: 1 3 4 5 9
    cout << "\n";

    // multiset: like set, but allows duplicates
    multiset<int> ms{5, 3, 1, 1, 5};
    cout << "count of 1 in multiset: " << ms.count(1) << "\n";

    // map: sorted key-value pairs, also a red-black tree -- O(log n) operations
    map<string, int> ages;
    ages["Alice"] = 30;
    ages["Bob"] = 25;
    ages["Charlie"] = 35;
    for (auto &[name, age] : ages) {   // iterates in sorted KEY order alphabetically
        cout << name << ":" << age << " ";
    }
    cout << "\n";

    // THE TRAP: operator[] on a map INSERTS a default-constructed value if the key
    // doesn't exist. Using ages["Dave"] just to check existence silently creates "Dave":0.
    // Use .find() or .count() to check existence WITHOUT modifying the map.
    if (ages.find("Dave") == ages.end()) {
        cout << "Dave not found (map unmodified)\n";
    }
    cout << "map size before risky access=" << ages.size() << "\n";
    ages["Dave"];  // this INSERTS Dave:0 into the map right now
    cout << "map size after ages[\"Dave\"]=" << ages.size() << " (Dave was just created!)\n";

    // unordered_set / unordered_map: hash table backed, O(1) average, O(n) worst case,
    // NO ordering guarantee at all
    unordered_map<string,int> fast_lookup;
    fast_lookup["x"] = 1;
    fast_lookup["y"] = 2;
    cout << "unordered_map lookup x=" << fast_lookup["x"] << "\n";

    // when to use which: need sorted order or range queries (lower_bound/upper_bound) -> set/map
    // need pure fastest lookup, don't care about order -> unordered_set/unordered_map

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 9. Ordered Range Queries — `lower_bound` / `upper_bound` on `set` and `map`

### The Why
This is the actual reason to pick `set`/`map` over `unordered_set`/`unordered_map` even though the hashed versions are faster on average: ordered containers let you ask "what's the smallest element ≥ X?" in O(log n). An unordered container cannot answer that question faster than O(n), because it has no order to binary-search over.

### Core Mechanism
- `s.lower_bound(x)` → iterator to the first element **≥ x**.
- `s.upper_bound(x)` → iterator to the first element **> x** (strictly greater — excludes x itself if present).
- Both are O(log n) member functions on `set`/`map` (not to be confused with the free functions of the same name in `<algorithm>`, covered next, which work on any sorted range including plain vectors).
- On a `map`, these operate on the **keys**, and the returned iterator's `->first`/`->second` give you the key/value pair.

### Common Pitfalls
- Confusing `lower_bound` and `upper_bound` — the mnemonic: "lower" finds the LOWEST valid position for inserting x while keeping sort order (so it includes x itself if present); "upper" finds the position strictly after all copies of x.


In [ ]:
%%writefile temp.cpp
#include <set>
#include <map>
#include <iostream>
using namespace std;

int main() {
    // set/map give you ordered range queries -- something unordered_ containers CANNOT do
    set<int> s{10, 20, 30, 40, 50};

    // lower_bound(x): iterator to first element >= x
    // upper_bound(x): iterator to first element > x
    auto lb = s.lower_bound(25);
    auto ub = s.upper_bound(25);
    cout << "lower_bound(25)=" << *lb << " upper_bound(25)=" << *ub << "\n";

    auto lbExact = s.lower_bound(30);
    cout << "lower_bound(30)=" << *lbExact << "\n";   // 30 itself, since it exists and is >= 30

    // erase by value or by iterator -- both O(log n)
    s.erase(30);
    for (int x : s) cout << x << " ";
    cout << "\n";

    // map has the same lower_bound/upper_bound, operating on keys
    map<int, string> m{{1,"a"}, {5,"b"}, {10,"c"}};
    auto it = m.lower_bound(3);
    cout << "first key >= 3: " << it->first << " -> " << it->second << "\n";

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 10. Iterators — Categories, Validity, and `advance`

### The Why
Iterators are the glue between containers and algorithms — every `<algorithm>` function takes iterators, not containers, which is exactly what lets `sort()` work on a `vector` and a plain array with the same code. But not all iterators support the same operations, and mixing that up is a compile error at best, undefined behavior at worst.

### Core Mechanism
- `[begin(), end())` is a half-open range — `end()` points ONE PAST the last valid element. Dereferencing `end()` is undefined behavior; the loop condition `it != end()` is what keeps you safe.
- `cbegin()`/`cend()` give `const_iterator`s — read-only, use them when you're not modifying the container, both for safety and to document intent.
- `rbegin()`/`rend()` give reverse iterators — `rbegin()` is the LAST element, `rend()` is one-before-the-first. Iterating with these walks the container backwards without you writing any reverse-indexing logic.
- **Iterator categories** determine what operations are legal:
  - *Random access* (`vector`, `deque`, `array`): supports `it + n`, `it[n]`, full arithmetic — O(1) jump to any offset.
  - *Bidirectional* (`list`, `set`, `map`): supports `++it` and `--it`, but NOT `it + n` — jumping n steps is O(n), one at a time.
  - *Forward only* (`forward_list`): supports `++it` only, never `--it`.
- `std::advance(it, n)` moves an iterator forward by n steps, generically — it works correctly (and at the best possible speed) regardless of which category the iterator belongs to.

### Common Pitfalls
- Trying `it + 5` on a `list` or `set` iterator — this won't compile, because those are bidirectional, not random access. Use `advance()` instead.
- Dereferencing `end()` off-by-one, usually from a loop condition written as `<=` instead of `!=`/`<`.


In [ ]:
%%writefile temp.cpp
#include <vector>
#include <list>
#include <iostream>
using namespace std;

int main() {
    vector<int> v{1, 2, 3, 4, 5};

    // begin()/end(): [begin, end) is a half-open range -- end() points ONE PAST the last element
    // dereferencing end() is undefined behavior
    for (auto it = v.begin(); it != v.end(); ++it) {
        cout << *it << " ";
    }
    cout << "\n";

    // const_iterator: read-only access, use when you won't modify elements
    for (auto it = v.cbegin(); it != v.cend(); ++it) {
        cout << *it << " ";
        // *it = 99;  // would NOT compile -- const_iterator forbids writes
    }
    cout << "\n";

    // reverse_iterator: rbegin() is the LAST element, rend() is one-before-the-first
    for (auto it = v.rbegin(); it != v.rend(); ++it) {
        cout << *it << " ";
    }
    cout << "\n";

    // iterator categories matter for algorithm compatibility:
    // vector/array/deque -> random access (it + 5 works, O(1))
    // list/set/map -> bidirectional (++it and --it work, but NOT it + 5)
    // forward_list -> forward only (++it works, --it does NOT)
    list<int> lst{1, 2, 3};
    auto lit = lst.begin();
    ++lit; ++lit;          // fine, bidirectional supports increment
    // lit = lit + 2;       // would NOT compile -- list iterators aren't random access
    cout << "list iterator after 2 increments: " << *lit << "\n";

    // std::advance works generically across ALL iterator categories
    auto lit2 = lst.begin();
    advance(lit2, 2);
    cout << "advance() result: " << *lit2 << "\n";

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 11. The `<algorithm>` Header

### The Why
This header is the reason you rarely hand-write a sort, a linear scan for a max, or a permutation generator in competitive C++ — it's all here, tested, and fast. Knowing what's available (and what its actual contract is) saves you from reinventing broken versions of these functions under time pressure.

### Core Mechanism
- `sort(begin, end)` — O(n log n), introsort (quicksort with a heapsort fallback to avoid quicksort's O(n²) worst case). Pass a third argument (comparator, e.g. `greater<int>()` or a lambda) to change the ordering.
- `stable_sort` — same idea, but **preserves the relative order of equal elements**. Use this when you're sorting by one key but want ties to keep their original order (e.g. sorting log entries by severity, but same-severity entries should stay in timestamp order).
- `binary_search(begin, end, val)` — returns `bool` only, requires the range to already be sorted, O(log n).
- `lower_bound`/`upper_bound` (the `<algorithm>` free-function versions) — work on ANY sorted range, including plain vectors, not just `set`/`map`.
- **Erase-remove idiom** (see Section 2): `v.erase(remove(v.begin(), v.end(), val), v.end())`.
- `reverse(begin, end)` — in-place, O(n).
- `unique(begin, end)` — removes **consecutive** duplicates only. If you want ALL duplicates gone, `sort()` first so duplicates become consecutive, then call `unique()`, then `erase()` the leftover tail (same two-step pattern as erase-remove).
- `accumulate(begin, end, initial)` (from `<numeric>`) — sums a range, or folds with a custom binary operation if you pass one.
- `min_element`/`max_element` — return an **iterator**, not a value. Dereference it (`*it`) to get the value; subtract `begin()` from it to get the index.
- `count`/`count_if` — count occurrences of a value, or count elements matching a predicate.
- `next_permutation(begin, end)` — rearranges the range in place to the next lexicographic permutation, returns `false` (and wraps to the smallest/sorted order) once you've cycled through all of them. The standard way to enumerate all permutations of a sorted starting sequence.

### Common Pitfalls
- Calling `binary_search`/`lower_bound`/`upper_bound` on an UNSORTED range — the result is undefined, not just "wrong," because these algorithms assume sortedness and don't check it.
- Expecting `unique()` alone to shrink the container — same trap as `remove()`, it needs a follow-up `erase()`.
- Forgetting `next_permutation` needs to START from the smallest (sorted ascending) sequence to enumerate ALL permutations — starting from an arbitrary order only gives you the permutations from that point onward.


In [ ]:
%%writefile temp.cpp
#include <vector>
#include <algorithm>
#include <numeric>
#include <iostream>
using namespace std;

int main() {
    vector<int> v{5, 3, 8, 1, 9, 2};

    // sort: O(n log n), introsort under the hood (quicksort + heapsort fallback)
    sort(v.begin(), v.end());
    for (int x : v) cout << x << " ";
    cout << "\n";

    // sort descending: pass greater<int>(), or sort ascending then reverse
    sort(v.begin(), v.end(), greater<int>());
    for (int x : v) cout << x << " ";
    cout << "\n";
    sort(v.begin(), v.end());  // back to ascending for the rest

    // binary_search: returns bool only, needs a SORTED range, O(log n)
    cout << "contains 8? " << binary_search(v.begin(), v.end(), 8) << "\n";

    // lower_bound / upper_bound work on plain sorted vectors too, not just set/map
    auto lb = lower_bound(v.begin(), v.end(), 5);
    cout << "first element >= 5: " << *lb << "\n";

    // stable_sort: like sort, but preserves relative order of equal elements -- needed
    // when sorting by one key but wanting ties to keep their original order
    vector<pair<int,char>> tagged{{2,'a'}, {1,'b'}, {2,'c'}, {1,'d'}};
    stable_sort(tagged.begin(), tagged.end(), [](auto &a, auto &b){ return a.first < b.first; });
    for (auto &[num, tag] : tagged) cout << "(" << num << "," << tag << ") ";
    cout << "\n";   // (1,b) (1,d) (2,a) (2,c) -- b before d, a before c, ties preserved

    // reverse: in-place, O(n)
    reverse(v.begin(), v.end());
    for (int x : v) cout << x << " ";
    cout << "\n";

    // unique: removes CONSECUTIVE duplicates only -- sort first if you want all duplicates gone
    vector<int> dups{1, 1, 2, 2, 2, 3, 1};
    dups.erase(unique(dups.begin(), dups.end()), dups.end());
    for (int x : dups) cout << x << " ";  // 1 2 3 1 -- the trailing 1 survives, wasn't consecutive
    cout << "\n";

    // accumulate: sum (or custom fold) over a range
    int sum = accumulate(v.begin(), v.end(), 0);
    cout << "sum=" << sum << "\n";

    // min_element / max_element return ITERATORS, not values -- dereference to get the value
    auto maxIt = max_element(v.begin(), v.end());
    cout << "max=" << *maxIt << " at index " << (maxIt - v.begin()) << "\n";

    // count / count_if
    cout << "count of 9: " << count(v.begin(), v.end(), 9) << "\n";
    int evenCount = count_if(v.begin(), v.end(), [](int x){ return x % 2 == 0; });
    cout << "even count: " << evenCount << "\n";

    // next_permutation: rearranges to the NEXT lexicographic permutation, returns false when
    // it wraps back to the smallest (sorted ascending) order
    vector<int> perm{1, 2, 3};
    cout << "permutations: ";
    do {
        for (int x : perm) cout << x;
        cout << " ";
    } while (next_permutation(perm.begin(), perm.end()));
    cout << "\n";

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 12. Lambdas and `std::function`

### The Why
Lambdas are how you pass custom behavior into `sort`, `priority_queue`, `count_if`, and dozens of other algorithms without writing a named function every time. Capture semantics are the one part that actually needs careful understanding — get them wrong and you get stale or dangling data silently.

### Core Mechanism
- Anatomy: `[captures](params) -> return_type { body }`. The return type is usually omittable — the compiler infers it from the `return` statement.
- **Capture by value `[x]`**: takes a snapshot of `x` at the moment the lambda is CREATED. Later changes to the original `x` do not affect the lambda's copy.
- **Capture by reference `[&x]`**: the lambda sees LIVE changes to `x` — because it's not a copy, it's an alias.
- `[=]` captures everything the lambda uses, by value. `[&]` captures everything it uses, by reference. Both are convenient but can be dangerous — `[&]` in something that outlives its captured variables (like a lambda stored and called later, after a local variable went out of scope) is a dangling reference bug.
- `mutable` allows a BY-VALUE capture to be modified inside the lambda body — this only changes the lambda's own private copy, never the original variable outside.
- `std::function<ReturnType(ArgTypes...)>` is a type-erased wrapper that can hold ANY callable matching that signature — a lambda, a plain function pointer, or a functor. Useful when you need to store or pass around callables without caring about their exact underlying type (which for a lambda is compiler-generated and unnameable).

### Common Pitfalls
- Using `[&]` in a lambda that's stored and called later, after the referenced local variable has already gone out of scope — this is a dangling reference, and it's one of the more common sources of “works sometimes, crashes sometimes” bugs.
- Assuming a by-value capture updates automatically — it's frozen at creation time, not live.


In [ ]:
%%writefile temp.cpp
#include <functional>
#include <iostream>
#include <vector>
#include <algorithm>
using namespace std;

int main() {
    // lambda anatomy: [captures](params) -> return_type { body }
    auto add = [](int a, int b) -> int { return a + b; };
    cout << add(3, 4) << "\n";

    // capture by value [x]: takes a SNAPSHOT at creation time, later changes to x don't affect it
    int x = 10;
    auto byValue = [x]() { return x; };
    x = 999;
    cout << "capture by value: " << byValue() << "\n";  // still 10

    // capture by reference [&x]: sees LIVE changes to x
    int y = 10;
    auto byRef = [&y]() { return y; };
    y = 999;
    cout << "capture by reference: " << byRef() << "\n";  // 999

    // [=] captures everything used by value, [&] captures everything used by reference
    int a = 1, b = 2;
    auto both = [=]() { return a + b; };  // both by value
    cout << "capture all by value: " << both() << "\n";

    // mutable: lets a by-value capture be modified INSIDE the lambda (doesn't affect the original)
    int counter = 0;
    auto incrementer = [counter]() mutable { counter++; return counter; };
    cout << incrementer() << " " << incrementer() << " " << incrementer() << "\n"; // 1 2 3
    cout << "original counter unchanged: " << counter << "\n"; // 0

    // std::function: type-erased wrapper that can hold ANY callable (lambda, function pointer, functor)
    // useful when you need to store callables of the same signature but different implementations
    function<int(int,int)> op;
    op = [](int a, int b) { return a + b; };
    cout << "function wrapping lambda: " << op(2, 3) << "\n";
    op = add;  // can also hold a plain function
    cout << "function wrapping plain function: " << op(2, 3) << "\n";

    // greater<T>() / less<T>(): function objects (functors) used constantly as comparators
    vector<int> v{3, 1, 4, 1, 5};
    sort(v.begin(), v.end(), greater<int>());
    for (int n : v) cout << n << " ";
    cout << "\n";

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 13. `std::bitset` and Custom Comparators for User-Defined Types

### The Why
`bitset` gives you a compact, fixed-size bit array — useful for subset representations in DP (bitmask DP) and compact `visited[]` arrays. Separately, the moment you put your OWN struct into a `set`/`map`/sorted `vector`, the compiler needs to know how to order your type — and it won't guess.

### Core Mechanism
- `bitset<N>` — N is a compile-time constant. Construct from an integer or a binary string.
- `.count()` — number of set (1) bits. `.set(i)` — turn bit i on. `.flip(i)` — toggle bit i. `.to_ulong()` — convert back to an integer.
- Indexing with `[]` works for both reading and writing individual bits.
- **Custom types in ordered containers:** `set<Point>` (or `map<Point, V>`, or `sort()`ing a `vector<Point>`) requires `Point` to define `operator<` — this is what the container uses to decide ordering. Without it, the code simply won't compile.
- The convention for multi-field ordering: compare the first field; if equal, fall through and compare the next field, and so on — exactly how `pair`'s built-in lexicographic comparison works.

### Common Pitfalls
- Putting a custom struct into a `set` or as a `map` key without defining `operator<`, and being confused by the resulting compiler error (which usually just says something like "no matching function" instead of clearly pointing at the missing operator).


In [ ]:
%%writefile temp.cpp
#include <bitset>
#include <iostream>
#include <set>
using namespace std;

struct Point {
    int x, y;
    // set/map need a way to ORDER elements -- provide operator< to define that order
    bool operator<(const Point &other) const {
        if (x != other.x) return x < other.x;
        return y < other.y;
    }
};

int main() {
    // bitset<N>: fixed-size sequence of bits, N known at compile time
    bitset<8> bits(42);              // 42 in binary = 00101010
    cout << bits << "\n";
    cout << "count of set bits: " << bits.count() << "\n";
    bits.set(0);                     // turn on bit 0
    bits.flip(1);                    // toggle bit 1
    cout << bits << "\n";
    cout << "as unsigned long: " << bits.to_ulong() << "\n";

    // bitset is often used as a compact visited[] array or a subset-representation in DP
    bitset<5> subset;
    subset[1] = 1;
    subset[3] = 1;
    cout << "subset representation: " << subset << "\n";

    // custom struct in an ordered container: MUST define operator< (or pass a comparator)
    set<Point> points;
    points.insert({2, 1});
    points.insert({1, 5});
    points.insert({1, 2});
    for (auto &p : points) cout << "(" << p.x << "," << p.y << ") ";
    cout << "\n";  // ordered by x first, then y -- exactly what operator< defines

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 14. Phase Checkpoint, Complexity Cheat Sheet, and Answer Key

### Complexity Cheat Sheet

| Container | Access | Insert/Erase (end) | Insert/Erase (middle) | Search | Notes |
|---|---|---|---|---|---|
| `vector` | O(1) | O(1) amortized | O(n) | O(n), O(log n) if sorted | Contiguous, cache-friendly default |
| `deque` | O(1) | O(1) both ends | O(n) | O(n) | Fast at both ends, unlike vector |
| `list` | O(n) | O(1) at ends | O(1) with iterator, O(n) to reach it | O(n) | No random access, `splice()` is O(1) |
| `forward_list` | O(n) | O(1) front only | O(1) with iterator | O(n) | Singly linked, minimal overhead |
| `set`/`map` | — | O(log n) | O(log n) | O(log n) | Sorted iteration, red-black tree |
| `unordered_set`/`map` | — | O(1) avg, O(n) worst | O(1) avg, O(n) worst | O(1) avg, O(n) worst | No ordering guarantee |
| `priority_queue` | O(1) top only | O(log n) | — | — | Max-heap by default |

### Checkpoint Questions
1. Why is `vector::push_back` described as O(1) "amortized" instead of just O(1)?
2. What's the actual difference between what `remove()` does and what `erase()` does, and why do you need both?
3. What happens if you write `if (myMap["key"] == someValue)` to check for a key that might not exist?
4. Why can't you write `it + 3` on a `list<int>::iterator`?
5. You need to sort a `vector<pair<int,char>>` by the int, but want entries with equal ints to keep their original relative order. Which sort function do you use?
6. When would you choose `unordered_map` over `map`, and when would that choice actively hurt you?

### Answer Key
1. A single `push_back` that triggers reallocation is O(n) (copy every element to new memory) — but reallocation only happens roughly every log₂(n) calls (capacity doubles each time), so averaged over many calls, the cost per call works out to O(1).
2. `remove()` moves the elements you want to KEEP to the front of the range and returns an iterator to the new logical end, but doesn't change the container's `size()`. `erase()` actually deletes the leftover tail and shrinks the container. You need both — `remove()` alone leaves garbage duplicate data at the end that's still "in" the vector.
3. It inserts `"key"` into the map with a default-constructed value (0, empty string, etc.) if it wasn't already present — silently mutating the map — before doing the comparison. Use `.find()` or `.count()` to check existence without this side effect.
4. `list` iterators are bidirectional, not random access — they only support `++`/`--`, one step at a time. There's no O(1) way to jump n steps in a linked list, so the `+` operator isn't provided; use `advance(it, 3)` instead, which does it correctly (in O(n) for a list).
5. `stable_sort` — regular `sort` doesn't guarantee ties keep their original relative order; `stable_sort` does.
6. Choose `unordered_map` when you need the fastest average lookup/insert and don't care about iteration order at all. It hurts you when you need sorted iteration, range queries (`lower_bound`/`upper_bound`), or when adversarial input could trigger worst-case O(n) hash collisions and you can't control the hash function.

### Mini Project
Using only what's covered in this notebook, implement a simple word-frequency counter:
1. Read a sentence into a `stringstream` and split it into words.
2. Count occurrences of each word using a `map<string,int>` (use `operator[]` correctly this time — it's the RIGHT tool for counting, since you want the default-insert-as-0 behavior here).
3. Copy the (word, count) pairs into a `vector<pair<string,int>>` and `sort` it by count descending, using a lambda comparator.
4. Print the top 3 most frequent words.

This exercises: `stringstream` splitting, `map` for counting, structured bindings, custom lambda comparators, and `vector` of `pair`.
